# 15 - ניתוח הרשת האוטובוסית בלבד

הגרף הארצי שנבנה ב-notebook `02_graph_construction` מערבב את כל אופני התחבורה שבפיד ה-GTFS הישראלי: אוטובוס, רכבת כבדה, רכבת קלה, רכבל, טרולייבוס ושירות מבוסס-ביקוש. אוטובוס הוא האופן הדומיננטי בפער ניכר - כ-97% מכלל הנסיעות המתוזמנות - ולכן מפתה להניח שרשת כל האופנים *היא* רשת האוטובוסים. ה-notebook הזה בונה את הגרף **האוטובוסי בלבד** מתוך הפיד הגולמי ובוחן את ההנחה הזו במקום לקבל אותה כמובנת מאליה.

באופן קונקרטי הוא: קורא בזרימה את `stop_times.txt` תוך שמירה רק על נסיעות שהקו שלהן מקיים `route_type = 3` (אוטובוס רגיל), בונה את גרף סמיכות הנסיעות האוטובוסי, מודד את המבנה הגלובלי שלו, מחשב degree / weighted degree / PageRank / betweenness מדגמי, מחלץ את נקודות החיתוך (articulation points) והגשרים (bridges) שלו, מדרג את תחנות האוטובוס המובילות, מריץ ניסוי הסרת צמתים ממוקדת מול אקראית, ולבסוף משווה את הגרף האוטובוסי מול גרף כל האופנים צומת אחר צומת - ובפרט, **אילו תחנות חדלות או מתחילות להיות cut vertices ברגע שמסירים את הרכבות ואת שאר האופנים המשניים**. אוטובוס מבוסס-ביקוש (`route_type = 715`) נספר ומדווח בנפרד ולא ממוזג לתוך רשת האוטובוסים, משום שמדובר במודל שירות שונה (14 קווים / 458 נסיעות) ומיזוגו היה משנה בחשאי את ההגדרה של "אוטובוס".

**שאלת המחקר:** האם רשת האוטובוסים ניתנת להחלפה מבנית ברשת הארצית הרב-אופנית, או שהשכבה הלא-אוטובוסית הקטנה (בעיקר רכבות) משנה באופן מהותי היכן נמצאות נקודות הכשל הבודדות?

## קלט

* `israel-public-transportation/routes.txt`, `trips.txt` - למיפוי כל `trip_id` לאופן GTFS (האופן מאוחסן ב-`routes.txt`; מבצעים join של `trips -> routes` לפי `route_id`).
* `israel-public-transportation/stop_times.txt` - 816 MB / 15.7M שורות, **אינו מנוהל ב-git**, מורד לפי הצורך מ-Google Drive על ידי תא שלהלן ונקרא **שורה אחר שורה**, ולעולם אינו נטען כטבלה.
* `outputs/nb/02_graph_construction/tables/nodes.csv` - מאפייני התחנות (`stop_id, stop_name, lat, lon, region, metro, stop_use_count`).
* `outputs/nb/02_graph_construction/tables/edges.csv` - הקטעים המכוונים של כל האופנים (`from_stop, to_stop, trip_frequency`), המשמשים כבסיס ההשוואה.
* *בדיקות צולבות אופציונליות, המדולגות עם הדפסת הודעה אם אינן קיימות:* `outputs/nb/03_descriptive_analysis/tables/articulation_points.csv` ו-`outputs/nb/04_centrality_analysis/tables/stop_metrics.csv`.

## Notebooks שחייבים לרוץ קודם

1. `01_data_preparation` (מפיק את טבלת התחנות המנוקה שבה משתמש 02),
2. `02_graph_construction` (**נדרש** - ה-notebook הזה קורא את `nodes.csv` ו-`edges.csv` שלו).
3. `03_descriptive_analysis` ו-`04_centrality_analysis` הם אופציונליים; הם משמשים אך ורק לבדיקה צולבת של מספרים שה-notebook הזה מחשב מחדש בעצמו.

## פלט (הכול תחת `outputs/nb/15_bus_network/`)

| נתיב | תוכן |
|---|---|
| `bus_graph_undirected.pkl` | `networkx.Graph` בפיקול - הרשת האוטובוסית בלבד, לא מכוונת, מאפיין הקשת `weight` = נסיעות אוטובוס לכל קטע |
| `bus_summary.json` | כל מספר מפתח שהופק כאן, בתוספת הקבועים שבהם נעשה שימוש |
| `tables/bus_station_metrics.csv` | `stop_id, stop_name, lat, lon, region, degree, weighted_degree, approx_betweenness, is_articulation_point` (+ תוספות) |
| `tables/bus_edges.csv` | הקטעים המכוונים של האוטובוסים, כך שאף שלב במורד הזרם לא יידרש לקרוא שוב בזרימה 816 MB |
| `tables/top_bus_stations.csv` | איחוד של N התחנות המובילות תחת ארבעה דירוגים |
| `tables/bus_articulation_points.csv`, `tables/bus_bridges.csv` | cut vertices ו-cut edges של האוטובוסים בלבד |
| `tables/bus_resilience.csv` | עקומות הסרת צמתים ממוקדת מול אקראית |
| `tables/bus_vs_allmode_summary.csv` | מבנה האוטובוסים ומבנה כל האופנים זה לצד זה |
| `tables/articulation_point_comparison.csv` | כל תחנה שהיא cut vertex ברשת אחת ולא באחרת |
| `tables/demand_responsive_summary.csv` | שכבת `route_type = 715`, המדווחת בנפרד |
| `figures/top_bus_stations.png`, `figures/bus_resilience_curves.png`, `figures/bus_vs_allmode.png` | שלושת האיורים |

דבר מחוץ ל-`outputs/nb/15_bus_network/` אינו נכתב. בתיקיות המצוטטות בדוח `outputs/tables`, `outputs/figures` ו-`outputs/rail` לא נוגעים כלל.

## 1. אתחול סביבת העבודה

התא שלהלן מאפשר להריץ את ה-notebook הן על עותק מקומי והן על Google Colab. הוא מגדיר את `_ensure(...)`, המתקין ב-pip רק את החבילות שאכן חסרות (כך שהרצה חוזרת של ה-notebook זולה), ואת `find_repo_root()`, המטפס כלפי מעלה מהתיקייה הנוכחית בחיפוש אחר תיקיית ה-GTFS, ואם לא מצא - משכפל את המאגר לתוך `/content`. לאחר מכן הוא מגדיר את `REPO`, `DATA` ו-`OUT` ויוצר את שורש הפלט של ה-notebook. כל תא מאוחר יותר מסתמך על שלושת הנתיבים הללו, ולכן זהו התא שחייב לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות השלב וידיות עלות

אנו מייבאים את הערימה המדעית, מקבעים את מבנה התיקיות של השלב הזה (`outputs/nb/15_bus_network/` עם `tables/` ו-`figures/`), ומרכזים כל הגדרה יקרה במקום אחד, כדי שהבודק יוכל להחליף זמן ריצה בדיוק בלי לחפש בתוך ה-notebook.

**מה עולה כל קבוע:**

* `K_BETWEENNESS = 300` - מספר מקורות הציר (pivots) עבור `networkx.betweenness_centrality(k=...)`. חישוב betweenness מדויק על גרף בן כ-29k צמתים היה דורש 29k סריקות מסלול קצר ביותר ממקור יחיד (שעות); 300 pivots אורכים בערך **1-3 דקות לכל גרף**. אנו משלמים זאת פעמיים - פעם עבור גרף האוטובוסים ופעם עבור גרף כל האופנים - משום שהשוואת ערכי האוטובוס המדגמיים שלנו מול טבלה שנדגמה *באופן שונה* ב-notebook אחר לא הייתה השוואה שוויונית. מדובר ב**אומדן**: ראש הדירוג יציב למדי ב-k=300, הזנב הארוך רועש. יש להעלות את הערך אם רוצים אומדני זנב הדוקים יותר.
* `COMPUTE_ALLMODE_BETWEENNESS = True` - יש לקבוע ל-`False` כדי לדלג על הרצת ה-betweenness השנייה (חוסך מחצית מהעלות, ומבטל את חלק ה-betweenness בהשוואה בין אוטובוס לכל האופנים).
* `RANDOM_TRIALS = 10` ו-`REMOVAL_COUNTS` - ניסוי הסרת הצמתים. כל זוג (אסטרטגיה, מספר הסרות) עולה מעבר אחד של רכיבי קשירות על הגרף השורד, כך שהסך הכול הוא `(3 ממוקדות + 10 אקראיות) x 10 נקודות` ~ 130 מעברים, הרבה פחות מדקה.
* `PROGRESS_EVERY = 2_000_000` - מרווח הדפסת ההתקדמות של מעבר הזרימה. קוסמטי.
* מעבר הזרימה עצמו הוא העלות האמיתית של ה-notebook הזה: **3-6 דקות** עבור קריאה סדרתית אחת של 15.7M שורות.
* `SEED = 42` מקבע את ה-pivots של ה-betweenness ואת סדרי ההסרה האקראיים, כך שהרצה חוזרת משחזרת את אותם המספרים.

In [ ]:
# --- Libraries, stage folders and cost knobs ------------------------------
_ensure('pandas', 'numpy', 'networkx', 'matplotlib', 'seaborn', 'scipy')

import csv, json, pickle, random, time
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', font_scale=1.05)
csv.field_size_limit(10_000_000)   # a handful of rows in stop_times.txt are very long

# --- GTFS mode codes (verified against this feed) -------------------------
BUS_ROUTE_TYPE = '3'      # ordinary bus  : 6,796 routes / 412,544 trips
DRT_ROUTE_TYPE = '715'    # demand-responsive bus : 14 routes / 458 trips

# --- Cost knobs (see the markdown above) ----------------------------------
K_BETWEENNESS = 300                  # betweenness pivots; ~1-3 min per graph
COMPUTE_ALLMODE_BETWEENNESS = True   # False -> skip the second betweenness run
SEED = 42                            # pivots + random removal orderings
RANDOM_TRIALS = 10                   # Monte-Carlo repetitions of random failure
REMOVAL_COUNTS = [0, 10, 25, 50, 100, 250, 500, 1000, 2000, 3000]
PROGRESS_EVERY = 2_000_000           # progress print interval of the streaming pass
FIG_DPI = 150                        # figure resolution
TOP_N = 20                           # rows in every top-N table / bar chart

# --- This stage's own output folder ---------------------------------------
STAGE = OUT / '15_bus_network'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print('networkx', nx.__version__, '| pandas', pd.__version__)
print('stage folder:', STAGE)

## 3. עיבוד תוויות בעברית

שמות התחנות בפיד ה-GTFS הישראלי הם בעברית ומספר איורים שלהלן מדפיסים אותם. Matplotlib אינה ממשת את האלגוריתם הדו-כיווני (bidirectional) של Unicode, ולכן טקסט מימין לשמאל מוצג הפוך ובלתי קריא. התא שלהלן מבצע monkey-patch חד-פעמי ל-`matplotlib.text.Text.set_text` כך שכל מחרוזת המכילה תווים עבריים מומרת לסדר תצוגה באמצעות `python-bidi` לפני ציורה, ובוחר גופן שאכן מכיל גליפים עבריים (Arial ב-Windows, DejaVu Sans בכל שאר הסביבות). הפעולה אידמפוטנטית - הרצה חוזרת לא תערים patches זה על זה. מאחר שה-patch גלובלי, יש להעביר ל-matplotlib מכאן והלאה מחרוזות עבריות **גולמיות**; קריאה ידנית נוספת ל-`fix_he()` תהפוך את הטקסט פעמיים. כל שאר הטקסט ב-notebook הוא באנגלית.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. טעינת התוצרים של notebook 02

שני דברים מגיעים מהשלבים הקודמים: **טבלת מאפייני התחנות** (`nodes.csv`), המעניקה לכל תחנת אוטובוס את שמה העברי, קואורדינטות, מחוז ואזור מטרופוליני, ו**רשימת הקשתות של כל האופנים** (`edges.csv`), שהיא הבסיס שמולו מושווית רשת האוטובוסים בסעיף 11.

תיקיות השלב מזוהות לפי הקידומת הדו-ספרתית שלהן (`OUT.glob('02*')`) ולא לפי slug מדויק, כך ששינוי שם תיקייה אינו שובר את ה-notebook, והתוצר עצמו מחופש רקורסיבית בתוך תיקיית השלב (הוא עשוי לשבת בשורש או תחת `tables/`). אם הוא חסר, `load_stage_table` זורקת `FileNotFoundError` המציינת בשמו את ה-notebook שיש להריץ קודם - נסיגה שקטה כאן הייתה מייצרת גרף חסר מאפיינים ומפות חסרות פשר עשרה תאים מאוחר יותר. `try_load_stage_table` היא הווריאנט הרך המשמש לבדיקות הצולבות האופציונליות מול notebooks 03 ו-04.

In [ ]:
# --- Locating artifacts written by earlier notebooks ----------------------
def find_stage(prefix, notebook_hint):
    """Return the stage folder whose name starts with `prefix` (e.g. '02')."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            f'No stage folder starting with "{prefix}" under {OUT} - '
            f'run notebook {notebook_hint} first.')
    return matches[0]


def find_artifact(stage_dir, filename):
    """Path of `filename` anywhere under a stage folder, or None if absent."""
    direct = stage_dir / filename
    if direct.exists():
        return direct
    matches = sorted(stage_dir.rglob(filename))
    return matches[0] if matches else None


def load_stage_table(prefix, filename, notebook_hint, **kwargs):
    """Read a CSV produced by an earlier stage; fail with an actionable message."""
    stage_dir = find_stage(prefix, notebook_hint)
    path = find_artifact(stage_dir, filename)
    if path is None:
        raise FileNotFoundError(
            f'{filename} not found under {stage_dir} - run notebook {notebook_hint} '
            f'first; it writes {filename}.')
    table = pd.read_csv(path, encoding='utf-8-sig', **kwargs)
    print(f'loaded {filename}: {len(table):,} rows  <-  {path}')
    return table


def try_load_stage_table(prefix, filename, notebook_hint, **kwargs):
    """Optional variant: returns None and prints a note instead of raising."""
    try:
        return load_stage_table(prefix, filename, notebook_hint, **kwargs)
    except FileNotFoundError as exc:
        print('OPTIONAL INPUT MISSING -', exc)
        return None


nodes_all = load_stage_table('02', 'nodes.csv', '02_graph_construction',
                             dtype={'stop_id': str})
edges_all = load_stage_table('02', 'edges.csv', '02_graph_construction',
                             dtype={'from_stop': str, 'to_stop': str})
nodes_all.head()

## 5. תלות בנתונים חיצוניים: `stop_times.txt`

הקובץ `stop_times.txt` שוקל 816 MB - הרבה מעל מגבלת גודל הקובץ של GitHub - ולכן הוא **אינו** נמצא במאגר. התא שלהלן מוריד אותו מ-Google Drive בהרצה הראשונה ומדלג על ההורדה אם הקובץ כבר קיים. זוהי התלות הרשתית החיצונית היחידה של ה-notebook. ההורדה אורכת מספר דקות בהרצת Colab ראשונה.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 6. אילו נסיעות הן נסיעות אוטובוס?

GTFS מאחסן את האופן על ה**קו** (route) ולא על הנסיעה, ולכן "נסיעת אוטובוס" חייבת להיגזר: קוראים את `routes.txt`, שומרים על `route_type = 3`, ואז שומרים כל `trip_id` ב-`trips.txt` שה-`route_id` שלו מצביע על אחד מהקווים הללו. אותו הדבר נעשה עבור `route_type = 715` (אוטובוס מבוסס-ביקוש), אותו אנו משאירים במכוון כשכבה *נפרדת*.

התא גם מדפיס את מלאי האופנים המלא (קווים ונסיעות לכל `route_type`), כך שהטענה על 97% מהנסיעות שבמבוא מאומתת מתוך הנתונים ולא נטענת סתם, ומדווח כמה נסיעות נושאות `route_id` שאינו מופיע כלל ב-`routes.txt` - מספר זה אמור להיות אפס, ואם אינו כזה, שיוך האופנים אינו שלם וכל ספירה שלהלן היא חסם תחתון.

In [ ]:
# --- Map every trip_id to its GTFS mode -----------------------------------
routes = pd.read_csv(DATA / 'routes.txt', dtype=str, encoding='utf-8-sig')
trips = pd.read_csv(DATA / 'trips.txt', dtype=str, encoding='utf-8-sig')

MODE_LABELS = {'0': 'tram / light rail', '2': 'rail', '3': 'bus',
               '5': 'cable tram', '8': 'trolleybus',
               '715': 'demand-responsive bus'}

route_type_of = dict(zip(routes['route_id'], routes['route_type']))
trips['route_type'] = trips['route_id'].map(route_type_of)
unmapped_trips = int(trips['route_type'].isna().sum())

inventory = (trips.groupby('route_type').size().rename('trips').to_frame()
             .join(routes.groupby('route_type').size().rename('routes'))
             .reset_index())
inventory['mode_label'] = inventory['route_type'].map(MODE_LABELS).fillna('other')
inventory['trip_share'] = (inventory['trips'] / inventory['trips'].sum()).round(4)
inventory = (inventory[['route_type', 'mode_label', 'routes', 'trips', 'trip_share']]
             .sort_values('trips', ascending=False).reset_index(drop=True))

bus_trip_ids = set(trips.loc[trips['route_type'] == BUS_ROUTE_TYPE, 'trip_id'])
drt_trip_ids = set(trips.loc[trips['route_type'] == DRT_ROUTE_TYPE, 'trip_id'])
if not bus_trip_ids:
    raise RuntimeError('No route_type=3 (bus) trips found - check routes.txt / trips.txt.')

# trip_id -> layer label used by the streaming pass below
TRIP_MODE = {tid: 'bus' for tid in bus_trip_ids}
TRIP_MODE.update({tid: 'drt' for tid in drt_trip_ids})

print(f'bus trips (route_type {BUS_ROUTE_TYPE})        : {len(bus_trip_ids):,} '
      f'({len(bus_trip_ids) / len(trips):.2%} of all trips)')
print(f'demand-responsive trips ({DRT_ROUTE_TYPE})     : {len(drt_trip_ids):,}')
print(f'trips with a route_id absent from routes.txt : {unmapped_trips:,}')
inventory

## 7. מעבר הזרימה על 15.7M שורות

זהו השלב היקר (**3-6 דקות**) והמקום היחיד שבו נוגעים בפיד בן 816 MB. `csv.reader` מחזיר שורה אחת בכל פעם; המצב היחיד הנשמר הוא מוני הקטעים (כ-50k רשומות), ספירות העצירה לכל תחנה, והנסיעה / התחנה / הרצף של השורה הקודמת. הזיכרון הוא `O(|E|)`, לעולם לא `O(rows)`.

כלל הקשת זהה לזה של notebook 02: אם השורה הנוכחית שייכת לאותה נסיעה כמו השורה הקודמת ושתי התחנות שונות, מגדילים את `W(prev_stop, stop)`. הסינון מוחל **לכל נסיעה**, לא לכל שורה - או שכל שורות הנסיעה נשמרות או שאף אחת מהן - כך ששורות שמורות עוקבות של נסיעה שמורה הן עדיין תחנות עוקבות של אותה נסיעה, והנחת הרציפות של notebook 02 עוברת ללא שינוי. נסיעות של אופנים אחרים מדולגות לאחר חיפוש יחיד במילון המבוצע פעם אחת לכל בלוק נסיעה, ולא פעם אחת לכל שורה.

שני מנגנוני הגנה של "כשל כן" נלווים ללא עלות נוספת: לולאות עצמיות (נסיעה המונה את אותה תחנה פעמיים ברצף) נספרות ומדולגות במקום להפוך לקשתות לולאה חסרות משמעות, ונסיגות של `stop_sequence` בתוך בלוק נסיעה נספרות. מספר נסיגות שאינו אפס פירושו שהפיד **אינו** ממוין לפי `(trip_id, stop_sequence)` והקטעים המופקים כאן יהיו שגויים בלי שיורגש; במקרה כזה התא מדפיס אזהרה בולטת.

יש לשים לב שאיננו מפרשים כאן כלל את `arrival_time` / `departure_time`. ה-notebook הזה הוא מחקר טופולוגי טהור; הסתייגות "שעות >= 24" של GTFS (`25:30:00` = 01:30 ביום השירות הבא) רלוונטית רק ל-notebooks של זמני הנסיעה.

In [ ]:
# --- One streaming pass over stop_times.txt, counting segments per mode ----
def stream_mode_edges(path, trip_mode, progress_every=PROGRESS_EVERY):
    """Count directed segments u->v separately for each mode label in `trip_mode`.

    Assumes stop_times.txt is sorted by (trip_id, stop_sequence); the assumption is
    re-checked over every kept row while we are already reading them.
    Returns (edge_counts, stop_calls, stats).
    """
    modes = sorted(set(trip_mode.values()))
    edge_counts = {m: defaultdict(int) for m in modes}
    stop_calls = {m: defaultdict(int) for m in modes}
    trips_seen = {m: set() for m in modes}
    rows_read = rows_kept = self_loops = seq_regressions = 0
    t0 = time.time()

    with open(path, encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        for field in ('trip_id', 'stop_id', 'stop_sequence'):
            if field not in header:
                raise ValueError(f'stop_times.txt has no {field} column: {header}')
        ti = header.index('trip_id')
        si = header.index('stop_id')
        qi = header.index('stop_sequence')

        prev_trip, prev_stop, prev_seq, mode = None, None, None, None
        for row in reader:
            rows_read += 1
            if progress_every and rows_read % progress_every == 0:
                print(f'    {rows_read:,} rows | bus segments so far '
                      f'{len(edge_counts["bus"]):,} | {time.time() - t0:,.0f}s')
            trip = row[ti]
            if trip != prev_trip:
                # new trip block: decide its mode once, reset the per-trip state
                prev_trip, prev_stop, prev_seq = trip, None, None
                mode = trip_mode.get(trip)
                if mode is not None:
                    trips_seen[mode].add(trip)
            if mode is None:
                continue                      # a trip of some other mode
            rows_kept += 1
            stop = row[si]
            stop_calls[mode][stop] += 1
            try:
                seq = int(row[qi])
            except (ValueError, IndexError):
                seq = None
            if prev_stop is not None:
                if prev_stop != stop:
                    edge_counts[mode][(prev_stop, stop)] += 1
                else:
                    self_loops += 1
                if seq is not None and prev_seq is not None and seq <= prev_seq:
                    seq_regressions += 1
            prev_stop, prev_seq = stop, seq

    stats = {
        'rows_scanned': rows_read,
        'rows_kept': rows_kept,
        'kept_share_of_feed': round(rows_kept / max(rows_read, 1), 4),
        'self_loops_skipped': self_loops,
        'stop_sequence_regressions': seq_regressions,
        'seconds': round(time.time() - t0, 1),
    }
    for m in modes:
        stats[f'{m}_trips_streamed'] = len(trips_seen[m])
        stats[f'{m}_directed_segments'] = len(edge_counts[m])
        stats[f'{m}_stops_touched'] = len(stop_calls[m])
    return edge_counts, stop_calls, stats


edge_counts, stop_calls, stream_stats = stream_mode_edges(STOP_TIMES, TRIP_MODE)
print(json.dumps(stream_stats, indent=2))
if stream_stats['stop_sequence_regressions']:
    print('\n' + '!' * 78)
    print('WARNING: stop_sequence regressions were found inside trip blocks.')
    print('The feed is not sorted by (trip_id, stop_sequence), so the segments built')
    print('below connect stops that are not actually consecutive. Sort the file first.')
    print('!' * 78)

## 8. מימוש גרף האוטובוסים

הקטעים שנספרו הופכים לאובייקטים של `networkx`: `D_bus`, גרף סמיכות הנסיעות המכוון (משקל הקשת = מספר נסיעות האוטובוס על אותו קטע), ו-`G_bus`, ההיטל הלא-מכוון שלו שבו שני כיווני הנסיעה **מסוכמים**, כך שמשקל לא-מכוון שומר על משמעותו הפיזית של "סך שירותי האוטובוס החוצים את הקישור הזה בכל אחד מהכיוונים". כל עבודת הקשירות שלהלן (רכיבים, cut vertices, הסרת צמתים) אינה תלוית כיוון ומשתמשת ב-`G_bus`.

מאפייני הצמתים - שם בעברית, קואורדינטות, מחוז, מטרופולין - מועתקים מ-`nodes.csv` של שלב 02. `_num` ו-`_txt` ממירים ערכים ריקים ו-`NaN` באופן בטוח, כך שקואורדינטה חסרה הופכת ל-`None` ולא ל-`NaN` שקט שהיה מצויר מאוחר יותר במיקום חסר פשר. תחנות אוטובוס שאינן נושאות מאפיינים נספרות ומדווחות; המספר אמור להיות ~0 משום שכל תחנת אוטובוס בעלת קטע היא גם צומת בגרף כל האופנים.

In [ ]:
# --- Build the bus graphs and attach stop attributes ----------------------
def build_graphs(counts):
    """Directed trip-adjacency graph plus its summed undirected projection."""
    D = nx.DiGraph()
    for (u, v), w in counts.items():
        D.add_edge(u, v, weight=int(w))
    G = nx.Graph()
    for u, v, data in D.edges(data=True):
        if G.has_edge(u, v):
            G[u][v]['weight'] += data['weight']
        else:
            G.add_edge(u, v, weight=data['weight'])
    return G, D


def _num(value):
    """Coerce to float; None for blanks, NaN or non-numeric input."""
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    return None if not np.isfinite(f) else f


def _txt(value):
    """Coerce to a plain string; NaN and None become an empty string."""
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ''
    return str(value)


ATTR = {str(r.get('stop_id')): {'stop_name': _txt(r.get('stop_name')),
                                'lat': _num(r.get('lat')),
                                'lon': _num(r.get('lon')),
                                'region': _txt(r.get('region')),
                                'metro': _txt(r.get('metro'))}
        for r in nodes_all.to_dict('records')}
DEFAULT_ATTR = {'stop_name': '', 'lat': None, 'lon': None, 'region': '', 'metro': ''}

G_bus, D_bus = build_graphs(edge_counts['bus'])
for graph in (G_bus, D_bus):
    for n in graph.nodes():
        graph.nodes[n].update(ATTR.get(n, DEFAULT_ATTR))

missing_attr = [n for n in G_bus.nodes() if n not in ATTR]
stops_without_segments = sorted(set(stop_calls['bus']) - set(G_bus.nodes()))

print(f'bus undirected : {G_bus.number_of_nodes():,} nodes, {G_bus.number_of_edges():,} edges')
print(f'bus directed   : {D_bus.number_of_nodes():,} nodes, {D_bus.number_of_edges():,} edges')
print(f'bus stops served by a trip but with no segment (excluded from V): '
      f'{len(stops_without_segments):,}')
print(f'bus nodes with no attributes in stage-02 nodes.csv: {len(missing_attr):,}')

## 9. המבנה הגלובלי של הרשת האוטובוסית בלבד

אותה מערכת מדדים תיאוריים ש-notebook 03 החיל על רשת כל האופנים, כעת על האוטובוס בלבד: גודל, צפיפות, סיכום דרגות, רכיבי קשירות והשיעור של התחנות ברכיב הגדול ביותר, ושני מבני הקשירות המקומית הקלאסיים - **נקודות חיתוך (articulation points)** (צמתים שהסרתם מנתקת את הגרף) ו**גשרים (bridges)** (קשתות שהסרתן מנתקת אותו). שניהם שגרות DFS בזמן ליניארי ב-`networkx` (רכיבים דו-קשירים בשיטת Hopcroft-Tarjan, פירוק שרשראות עבור גשרים) ומסתיימים בשניות גם בסדר גודל כזה. `articulation_points` עטוף ב-`set` משום שגרסאות ישנות יותר של `networkx` עלולות לפלוט את אותו cut vertex יותר מפעם אחת, מה שהיה מנפח את הספירה.

מספרים אלו הם העמודה השמאלית בהשוואה שבין אוטובוס לכל האופנים בסעיף 11.

In [ ]:
# --- Global structure of the bus network ----------------------------------
def structure_summary(G, D=None):
    """Size / sparsity / fragmentation / single-point-of-failure counts."""
    degrees = np.array([d for _, d in G.degree()])
    components = sorted(nx.connected_components(G), key=len, reverse=True)
    largest = len(components[0])
    ap = set(nx.articulation_points(G))
    br = list(nx.bridges(G))
    summary = {
        'nodes': G.number_of_nodes(),
        'edges_undirected': G.number_of_edges(),
        'edges_directed': D.number_of_edges() if D is not None else None,
        'density': round(nx.density(G), 6),
        'avg_degree': round(float(degrees.mean()), 3),
        'median_degree': float(np.median(degrees)),
        'max_degree': int(degrees.max()),
        'share_degree_le_2': round(float((degrees <= 2).mean()), 4),
        'connected_components': len(components),
        'largest_component_nodes': largest,
        'largest_component_share': round(largest / G.number_of_nodes(), 4),
        'articulation_points': len(ap),
        'articulation_point_share': round(len(ap) / G.number_of_nodes(), 4),
        'bridges': len(br),
        'bridge_share': round(len(br) / G.number_of_edges(), 4),
    }
    return summary, ap, br


bus_stats, ap_bus, bridges_bus = structure_summary(G_bus, D_bus)
deg_bus = dict(G_bus.degree())

for key, value in bus_stats.items():
    print(f'{key:28s} {value}')
pd.DataFrame([bus_stats])

## 10. Centrality על רשת האוטובוסים

ארבעה מדדים, שכל אחד מהם עונה על שאלה אחרת לגבי תחנת אוטובוס:

* **Degree** - כמה תחנות נבדלות נמצאות במרחק קטע אחד. קשירות מקומית טהורה.
* **Weighted degree** - אותו הדבר, משוקלל בנסיעות אוטובוס, כלומר כמה *נפח שירות* נוגע בתחנה.
* **PageRank** על הגרף המכוון והמשוקלל - חשיבות במובן של "היכן מצטבר זרם השירות", עמיד בפני ריבוי התחנות בעלות דרגה 2 לאורך קו.
* **Betweenness מדגמי** - איזה שיעור מהמסלולים הקצרים ביותר עובר דרך התחנה. זהו המדד המזהה צווארי בקבוק של מעברים ולא מסופים עמוסים.

**יש להיות כנים לגבי ה-betweenness.** betweenness מדויק דורש סריקת מסלולים קצרים ביותר מכל אחד מכ-29k הצמתים. אנו משתמשים ב-`K_BETWEENNESS = 300` pivots אקראיים (מקובעים על ידי `SEED`), וזהו **אומדן מדגמי**: ראש הדירוג יציב למדי, אך ערכים בודדים - במיוחד ערכים קטנים עמוק בזנב - נושאים שגיאת דגימה ממשית, ואין להתייחס לשתי תחנות בעלות אומדנים כמעט זהים כאילו סדרן אמין. משום כך העמודה נקראת `approx_betweenness`, בהתאם למוסכמה של notebook 04. המדד גם *מבוסס קפיצות* (`weight=None`): מסלול קצר ביותר הוא מספר הקטעים המזערי, לא הנסיעה המהירה ביותר, משום שזמני נסיעה נכנסים לפרויקט רק ב-notebook 18.

התוצאה נכתבת ל-`tables/bus_station_metrics.csv`, הקובץ שעליו נשענים notebooks מאוחרים יותר.

In [ ]:
# --- Centrality on the bus graph (the expensive part is betweenness) ------
t0 = time.time()
wdeg_bus = dict(G_bus.degree(weight='weight'))
pagerank_bus = nx.pagerank(D_bus, weight='weight')
k_bus = min(K_BETWEENNESS, G_bus.number_of_nodes())
btw_bus = nx.betweenness_centrality(G_bus, k=k_bus, seed=SEED,
                                    normalized=True, weight=None)
print(f'centrality computed in {time.time() - t0:,.0f}s '
      f'(betweenness pivots: {k_bus} of {G_bus.number_of_nodes():,} nodes)')

bus_metrics = pd.DataFrame([{
    'stop_id': n,
    'stop_name': G_bus.nodes[n]['stop_name'],
    'lat': G_bus.nodes[n]['lat'],
    'lon': G_bus.nodes[n]['lon'],
    'region': G_bus.nodes[n]['region'],
    'degree': int(deg_bus[n]),
    'weighted_degree': int(wdeg_bus[n]),
    'approx_betweenness': float(btw_bus[n]),
    'is_articulation_point': n in ap_bus,
    'metro': G_bus.nodes[n]['metro'],
    'pagerank': float(pagerank_bus[n]),
    'in_degree': int(D_bus.in_degree(n)),
    'out_degree': int(D_bus.out_degree(n)),
    'bus_stop_calls': int(stop_calls['bus'].get(n, 0)),
} for n in G_bus.nodes()])

for col in ('degree', 'weighted_degree', 'approx_betweenness', 'pagerank'):
    bus_metrics['rank_' + col] = (bus_metrics[col]
                                  .rank(ascending=False, method='min').astype(int))

bus_metrics = (bus_metrics.sort_values('approx_betweenness', ascending=False)
               .reset_index(drop=True))
bus_metrics.to_csv(TABLES / 'bus_station_metrics.csv', index=False, encoding='utf-8-sig')
print('saved:', TABLES / 'bus_station_metrics.csv')
bus_metrics.head(10)

## 11. תחנות האוטובוס המובילות

מדדים שונים ממנים תחנות שונות, ולכן במקום לבחור דירוג אחד אנו לוקחים את ה**איחוד** של `TOP_N` המובילות תחת כל ארבעת המדדים ומציגים היכן ממוקמת כל אחת מהן בכל מדד. תחנה שנמצאת ב-top-20 בכל ארבעת המדדים היא hub אמיתי; תחנה שנמצאת ב-top-20 רק ב-betweenness היא צוואר בקבוק הנושא בעצמו מעט שירות - סוג שונה מאוד של קריטיות, והשברירי יותר מבין השניים.

האיור מציג את שני הדירוגים הניתנים לפרשנות בקלות הרבה ביותר זה לצד זה: weighted degree (נפח שירות) ו-betweenness מדגמי (צוואר בקבוק במסלולים). שמות בעברית מצוירים דרך ה-bidi patch שהותקן בסעיף 3.

In [ ]:
# --- Top bus stations under four rankings ---------------------------------
rank_cols = ['rank_degree', 'rank_weighted_degree', 'rank_approx_betweenness',
             'rank_pagerank']
top_mask = (bus_metrics[rank_cols] <= TOP_N).any(axis=1)
top_table = (bus_metrics[top_mask]
             .sort_values('rank_approx_betweenness')
             [['stop_id', 'stop_name', 'region', 'metro', 'degree', 'weighted_degree',
               'approx_betweenness', 'pagerank', 'is_articulation_point'] + rank_cols]
             .reset_index(drop=True))
top_table.to_csv(TABLES / 'top_bus_stations.csv', index=False, encoding='utf-8-sig')
print(f'{len(top_table)} distinct stations appear in at least one top-{TOP_N} list')
print(f"of which {int(top_table['is_articulation_point'].sum())} are also bus cut vertices")

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
panels = [('weighted_degree', 'Bus trips on incident segments', '#2563eb'),
          ('approx_betweenness', f'Sampled betweenness (k={k_bus})', '#7c3aed')]
for ax, (col, xlabel, colour) in zip(axes, panels):
    panel = bus_metrics.sort_values(col, ascending=False).head(TOP_N).iloc[::-1]
    labels = [f'{name} ({sid})' for name, sid in zip(panel['stop_name'], panel['stop_id'])]
    ax.barh(range(len(panel)), panel[col].to_numpy(), color=colour)
    ax.set_yticks(range(len(panel)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel(xlabel)
    ax.set_title(f'Top {TOP_N} bus stations by {col.replace("_", " ")}')
plt.tight_layout()
plt.savefig(FIGURES / 'top_bus_stations.png', dpi=FIG_DPI)
plt.show()

top_table.head(TOP_N)

## 12. Cut vertices ו-cut edges של האוטובוסים

אלו הן נקודות הכשל הבודדות המבניות של רשת האוטובוסים: **נקודת חיתוך (articulation point)** היא תחנה שסגירתה מפצלת את הרכיב שלה לשניים, ו**גשר (bridge)** הוא קטע המהווה את הקישור היחיד בין שני חלקי הרשת. שתי הטבלאות מיוצאות במלואן, מועשרות בשם / קואורדינטות / מחוז / דרגה, כדי שהדוח ו-notebook השוואת העדשות המאוחר יותר יוכלו להצליב אליהן.

שתי הסתייגויות מוצהרות מראש, אותן הסתייגויות שחלות ב-notebook 03. ראשית, זוהי תפיסה **טופולוגית** טהורה של כשל - היא מתעלמת מכמה נוסעים משתמשים בקטע. שנית, נקודות חיתוך רבות הן תחנות בעלות דרגה נמוכה על ענף ללא מוצא, שבהן "מנתק את הרשת" משמעו "מותיר שלוש תחנות מבודדות בסוף קו"; מיון לפי דרגה, כפי שאנו עושים כאן, הוא מה שמפריד בין אלו לבין ה-hubs שאובדנם היה מבודד תת-רשת שלמה.

In [ ]:
# --- Export bus cut vertices and cut edges --------------------------------
ap_table = (bus_metrics[bus_metrics['is_articulation_point']]
            [['stop_id', 'stop_name', 'lat', 'lon', 'region', 'metro',
              'degree', 'weighted_degree', 'approx_betweenness']]
            .sort_values('degree', ascending=False).reset_index(drop=True))
ap_table.to_csv(TABLES / 'bus_articulation_points.csv', index=False, encoding='utf-8-sig')

name_of = {n: (G_bus.nodes[n].get('stop_name') or n) for n in G_bus.nodes()}
bridge_table = pd.DataFrame([{
    'from_stop': u,
    'to_stop': v,
    'from_name': name_of[u],
    'to_name': name_of[v],
    'trip_frequency': int(G_bus[u][v]['weight']),
} for u, v in bridges_bus]).sort_values('trip_frequency', ascending=False).reset_index(drop=True)
bridge_table.to_csv(TABLES / 'bus_bridges.csv', index=False, encoding='utf-8-sig')

print(f"bus articulation points : {len(ap_table):,} "
      f"({len(ap_table) / G_bus.number_of_nodes():.2%} of bus stops)")
print(f"bus bridges             : {len(bridge_table):,} "
      f"({len(bridge_table) / G_bus.number_of_edges():.2%} of undirected segments)")
print(f"median degree of a bus cut vertex: {ap_table['degree'].median():.0f}")
print('\nHighest-degree bus cut vertices:')
display(ap_table.head(TOP_N))
print('Busiest bus bridges:')
bridge_table.head(TOP_N)

## 13. חוסן: תקיפה ממוקדת מול כשל אקראי

אנו מוחקים תחנות מגרף האוטובוסים וצופים ב**רכיב הקשירות הגדול ביותר** מתכווץ. הגודל המדווח הוא גודל הרכיב השורד הגדול ביותר כשיעור ממספר הצמתים **המקורי**, כך שהעקומה משקפת גם את התחנות שמחקנו וגם את אלו שהן בודדו.

ארבעה סדרי הסרה מושווים:

* `degree` - דרגה גבוהה תחילה (שוויונות מוכרעים לפי weighted degree),
* `approx_betweenness` - betweenness מדגמי גבוה תחילה,
* `articulation_first` - cut vertices תחילה, ממוינים לפי דרגה, ולאחר מכן כל השאר לפי דרגה,
* `random` - סדר אקראי אחיד, ממוצע על פני `RANDOM_TRIALS` ניסויים בלתי תלויים, עם רישום סטיית התקן בין הניסויים.

**שתי מגבלות כנות.** (1) הדירוגים הממוקדים מחושבים **פעם אחת, על הגרף השלם, ולעולם אינם מחושבים מחדש** ככל שצמתים נעלמים. תוקף אדפטיבי אמיתי היה מחשב מחדש לאחר כל הסרה וגורם נזק רב יותר; לפיכך העקומות הממוקדות שלנו הן חסם תחתון לנזק שניתן להשיג. (2) סדר ה-betweenness יורש את שגיאת הדגימה של `K_BETWEENNESS`, ולכן זהו סדר *טוב*, לא סדר שהוכח כ*מיטבי*. שתי הבחירות הן הזולות, ושתיהן מטות את התוצאה נגד המסקנה שלנו עצמנו, וזהו הכיוון הבטוח.

In [ ]:
# --- Targeted vs random node removal --------------------------------------
def resilience_curve(graph, order, counts):
    """Largest-component share (of the ORIGINAL node count) after removing order[:c]."""
    n0 = graph.number_of_nodes()
    all_nodes = set(graph.nodes())
    shares = []
    for c in counts:
        remaining = all_nodes - set(order[:c])
        if not remaining:
            shares.append(0.0)
            continue
        sizes = [len(comp) for comp
                 in nx.connected_components(graph.subgraph(remaining))]
        shares.append(max(sizes) / n0)
    return shares


strategies = {
    'degree': bus_metrics.sort_values(['degree', 'weighted_degree'],
                                      ascending=False)['stop_id'].tolist(),
    'approx_betweenness': bus_metrics.sort_values('approx_betweenness',
                                                  ascending=False)['stop_id'].tolist(),
    'articulation_first': bus_metrics.sort_values(['is_articulation_point', 'degree'],
                                                  ascending=False)['stop_id'].tolist(),
}

t0 = time.time()
rows = []
for name, order in strategies.items():
    for c, share in zip(REMOVAL_COUNTS, resilience_curve(G_bus, order, REMOVAL_COUNTS)):
        rows.append({'strategy': name, 'removed': c,
                     'largest_component_share': round(share, 5),
                     'std_across_trials': 0.0})

rng = random.Random(SEED)
node_list = list(G_bus.nodes())
random_curves = []
for _ in range(RANDOM_TRIALS):
    shuffled = node_list[:]
    rng.shuffle(shuffled)
    random_curves.append(resilience_curve(G_bus, shuffled, REMOVAL_COUNTS))
random_mean = np.mean(random_curves, axis=0)
random_std = np.std(random_curves, axis=0)
for c, m, s in zip(REMOVAL_COUNTS, random_mean, random_std):
    rows.append({'strategy': 'random', 'removed': c,
                 'largest_component_share': round(float(m), 5),
                 'std_across_trials': round(float(s), 5)})

resilience = pd.DataFrame(rows)
resilience.to_csv(TABLES / 'bus_resilience.csv', index=False, encoding='utf-8-sig')
print(f'removal experiment finished in {time.time() - t0:,.0f}s')

fig, ax = plt.subplots(figsize=(10, 6))
colours = {'degree': '#2563eb', 'approx_betweenness': '#7c3aed',
           'articulation_first': '#dc2626', 'random': '#64748b'}
for name, grp in resilience.groupby('strategy'):
    grp = grp.sort_values('removed')
    ax.plot(grp['removed'], grp['largest_component_share'], marker='o',
            color=colours.get(name), label=name)
    if name == 'random':
        ax.fill_between(grp['removed'],
                        grp['largest_component_share'] - grp['std_across_trials'],
                        grp['largest_component_share'] + grp['std_across_trials'],
                        color=colours['random'], alpha=0.25)
ax.set_xlabel('Bus stations removed')
ax.set_ylabel('Largest component / original number of stations')
ax.set_title('Bus network under targeted attack and random failure\n'
             f'(static rankings, {RANDOM_TRIALS} random trials, shaded band = 1 sd)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'bus_resilience_curves.png', dpi=FIG_DPI)
plt.show()

resilience.pivot(index='removed', columns='strategy',
                 values='largest_component_share')

## 14. בנייה מחדש של רשת כל האופנים כבסיס השוואה

כדי לענות על השאלה "האם אוטובוס ניתן להחלפה ברשת כולה?" יש למדוד את שתי הרשתות **באותו האופן**. לכן, במקום לצטט את המספרים של notebook 03, אנו בונים מחדש את גרף כל האופנים מתוך `edges.csv` של שלב 02 (52k קשתות - כמה שניות) ומריצים מחדש בדיוק את אותה פונקציית `structure_summary`, בתוספת אומדן betweenness עם **אותו** `k` ו**אותו** seed. השוואת ה-betweenness המדגמי שלנו לאוטובוסים מול עמודה שנדגמה באופן שונה ב-notebook אחר הייתה מערבבת הבדל אמיתי עם רעש דגימה.

הרצת ה-betweenness כאן היא השלב היקר השני (1-3 דקות); יש לקבוע `COMPUTE_ALLMODE_BETWEENNESS = False` בסעיף 2 כדי לדלג עליה.

היכן ש-notebooks 03 ו-04 זמינים אנו משתמשים בהם כ**בדיקה צולבת בלתי תלויה** לחישוב מחדש שלנו (התאמת מספר ה-cut vertices, ומתאם הדירוגים בין מדגם ה-betweenness שלנו לכל האופנים לבין שלהם). אם הם אינם קיימים, ה-notebook מדפיס הודעה וממשיך - הם אינם נדרשים.

In [ ]:
# --- Rebuild and re-measure the all-mode network --------------------------
D_all = nx.DiGraph()
for rec in edges_all.to_dict('records'):
    D_all.add_edge(str(rec['from_stop']), str(rec['to_stop']),
                   weight=int(rec.get('trip_frequency', 1)))
G_all = nx.Graph()
for u, v, data in D_all.edges(data=True):
    if G_all.has_edge(u, v):
        G_all[u][v]['weight'] += data['weight']
    else:
        G_all.add_edge(u, v, weight=data['weight'])
for n in G_all.nodes():
    G_all.nodes[n].update(ATTR.get(n, DEFAULT_ATTR))

all_stats, ap_all, bridges_all = structure_summary(G_all, D_all)
deg_all = dict(G_all.degree())

btw_all = None
if COMPUTE_ALLMODE_BETWEENNESS:
    t0 = time.time()
    btw_all = nx.betweenness_centrality(G_all, k=min(K_BETWEENNESS, G_all.number_of_nodes()),
                                        seed=SEED, normalized=True, weight=None)
    print(f'all-mode betweenness computed in {time.time() - t0:,.0f}s')
else:
    print('all-mode betweenness skipped (COMPUTE_ALLMODE_BETWEENNESS = False)')

# --- optional cross-checks against notebooks 03 and 04 --------------------
ap03 = try_load_stage_table('03', 'articulation_points.csv', '03_descriptive_analysis',
                            dtype={'stop_id': str})
if ap03 is not None:
    ap03_ids = set(ap03['stop_id'])
    print(f'cross-check vs notebook 03: it lists {len(ap03_ids):,} cut vertices, '
          f'we recomputed {len(ap_all):,}; {len(ap03_ids & ap_all):,} agree')

metrics04 = try_load_stage_table('04', 'stop_metrics.csv', '04_centrality_analysis',
                                 dtype={'stop_id': str})
if metrics04 is not None and btw_all is not None and 'approx_betweenness' in metrics04:
    joint = metrics04[['stop_id', 'approx_betweenness']].dropna()
    joint = joint[joint['stop_id'].isin(btw_all)]
    rho04, _ = spearmanr(joint['approx_betweenness'],
                         [btw_all[s] for s in joint['stop_id']])
    print(f'cross-check vs notebook 04: Spearman rho between its all-mode '
          f'approx_betweenness and ours = {rho04:.3f} over {len(joint):,} stops '
          '(two independent samples of the same quantity, so <1 is expected)')

comparison = pd.DataFrame({'metric': list(bus_stats.keys()),
                           'bus_only': list(bus_stats.values()),
                           'all_modes': [all_stats[k] for k in bus_stats]})
comparison['bus_share_of_all'] = [
    round(b / a, 4) if isinstance(b, (int, float)) and isinstance(a, (int, float)) and a
    else None
    for b, a in zip(comparison['bus_only'], comparison['all_modes'])]
comparison.to_csv(TABLES / 'bus_vs_allmode_summary.csv', index=False, encoding='utf-8-sig')
print('\nsaved:', TABLES / 'bus_vs_allmode_summary.csv')
comparison

## 15. האם הסרת הרכבות משנה אילו תחנות הן cut vertices?

זוהי השאלה שלשמה קיים ה-notebook הזה, והתשובה אינה מובנת מאליה גם אם האוטובוס מהווה 97% מהנסיעות: קישורי הרכבת מעטים אך *ארוכים*, ומחברים ערים שרשת האוטובוסים עשויה לחבר רק דרך מסדרון יחיד. מחיקת מסלול עודף יכולה ליצור cut vertices חדשים, ולכן לרשת האוטובוסים בלבד יכולים להיות בקלות cut vertices שאינם קיימים ברשת כל האופנים.

כל תחנה שהיא cut vertex לפחות באחת משתי הרשתות מסווגת לאחת מארבע קטגוריות:

* `both` - cut vertex בשתי הרשתות; השכבה הלא-אוטובוסית אינה רלוונטית עבורה,
* `bus_only` - **אינה** cut vertex ברמה הארצית אך **כן** ברשת האוטובוסים בלבד: כאן הרכבת / הרכבת הקלה סיפקה את העודפות, ותצוגת כל האופנים *מסתירה* פגיעות אוטובוסית,
* `all_modes_only` - cut vertex ברמה הארצית אך לא ברשת האוטובוסים בלבד: היא ניתקה משהו שאליו מגיעים באופן לא-אוטובוסי,
* `absent_from_bus_network` - cut vertex של כל האופנים שאינו כלל תחנת אוטובוס (תחנת רכבת בלבד או רכבת קלה בלבד).

אנו מדווחים גם על חפיפת Jaccard בין שתי קבוצות ה-cut vertices, על מתאם Spearman בין דרגת האוטובוס לדרגת כל האופנים על פני התחנות המשותפות, ועל חפיפת ה-top-50 בין שני דירוגי ה-betweenness. שלושת המספרים הללו יחד אומרים עד כמה קרוב עוקבת רשת האוטובוסים אחר הרשת הארצית - נמדד, לא מונח.

In [ ]:
# --- Cut-vertex and ranking comparison ------------------------------------
bus_nodes = set(G_bus.nodes())
all_nodes = set(G_all.nodes())
shared_nodes = bus_nodes & all_nodes
bus_only_nodes = bus_nodes - all_nodes
non_bus_nodes = all_nodes - bus_nodes

ap_both = ap_bus & ap_all
ap_bus_only = ap_bus - ap_all
ap_all_only = (ap_all & bus_nodes) - ap_bus
ap_absent = ap_all - bus_nodes
jaccard_ap = len(ap_both) / max(len(ap_bus | ap_all), 1)


def _status(node):
    if node in ap_both:
        return 'both'
    if node in ap_bus_only:
        return 'bus_only'
    if node in ap_absent:
        return 'absent_from_bus_network'
    return 'all_modes_only'


ap_compare = pd.DataFrame([{
    'stop_id': n,
    'stop_name': (G_bus.nodes[n]['stop_name'] if n in bus_nodes
                  else G_all.nodes[n]['stop_name']),
    'region': (G_bus.nodes[n]['region'] if n in bus_nodes else G_all.nodes[n]['region']),
    'status': _status(n),
    'degree_bus': int(deg_bus.get(n, 0)),
    'degree_all_modes': int(deg_all.get(n, 0)),
    'is_articulation_point_bus': n in ap_bus,
    'is_articulation_point_all_modes': n in ap_all,
} for n in sorted(ap_bus | ap_all)])
ap_compare = ap_compare.sort_values(['status', 'degree_all_modes', 'degree_bus'],
                                    ascending=[True, False, False]).reset_index(drop=True)
ap_compare.to_csv(TABLES / 'articulation_point_comparison.csv', index=False,
                  encoding='utf-8-sig')

# degree agreement over the stops both networks contain
shared_sorted = sorted(shared_nodes)
rho_degree, _ = spearmanr([deg_bus[n] for n in shared_sorted],
                          [deg_all[n] for n in shared_sorted])
identical_degree = sum(1 for n in shared_sorted if deg_bus[n] == deg_all[n])

# undirected edge overlap
bus_edge_set = {tuple(sorted(e)) for e in G_bus.edges()}
all_edge_set = {tuple(sorted(e)) for e in G_all.edges()}

overlap = {
    'bus_nodes': len(bus_nodes),
    'all_mode_nodes': len(all_nodes),
    'shared_nodes': len(shared_nodes),
    'nodes_in_bus_not_in_all_modes': len(bus_only_nodes),
    'nodes_in_all_modes_not_in_bus': len(non_bus_nodes),
    'node_coverage_of_all_modes': round(len(shared_nodes) / len(all_nodes), 4),
    'undirected_edge_coverage_of_all_modes': round(
        len(bus_edge_set & all_edge_set) / len(all_edge_set), 4),
    'edges_in_bus_not_in_all_modes': len(bus_edge_set - all_edge_set),
    'spearman_degree_bus_vs_all_modes': round(float(rho_degree), 4),
    'stops_with_identical_degree': identical_degree,
    'share_with_identical_degree': round(identical_degree / len(shared_sorted), 4),
    'articulation_points_bus': len(ap_bus),
    'articulation_points_all_modes': len(ap_all),
    'articulation_points_both': len(ap_both),
    'articulation_points_bus_only': len(ap_bus_only),
    'articulation_points_all_modes_only': len(ap_all_only),
    'articulation_points_absent_from_bus': len(ap_absent),
    'articulation_point_jaccard': round(jaccard_ap, 4),
}

if btw_all is not None:
    top50_bus = set(bus_metrics.nsmallest(50, 'rank_approx_betweenness')['stop_id'])
    top50_all = {s for s, _ in sorted(btw_all.items(), key=lambda kv: kv[1],
                                      reverse=True)[:50]}
    rho_btw, _ = spearmanr([btw_bus[n] for n in shared_sorted],
                           [btw_all[n] for n in shared_sorted])
    overlap['spearman_betweenness_bus_vs_all_modes'] = round(float(rho_btw), 4)
    overlap['top50_betweenness_overlap'] = len(top50_bus & top50_all)

for key, value in overlap.items():
    print(f'{key:42s} {value}')
print('\nsaved:', TABLES / 'articulation_point_comparison.csv')
print('\nCut vertices that exist ONLY once the non-bus layer is removed '
      '(rail was the redundancy):')
display(ap_compare[ap_compare['status'] == 'bus_only'].head(TOP_N))
print('Cut vertices that stop being cut vertices on bus alone:')
ap_compare[ap_compare['status'] == 'all_modes_only'].head(TOP_N)

## 16. שלוש תמונות של אותה השוואה

* **שמאל** - דרגת האוטובוס מול דרגת כל האופנים עבור כל תחנה משותפת, עם הישר `y = x`. נקודות על הישר הן תחנות שהשכבה הלא-אוטובוסית אינה נוגעת בהן כלל; נקודות מעליו הן תחנות המרוויחות שכנים מהרכבת / הרכבת הקלה. מאחר שלרוב התחנות דרגות קטנות, הצירים לוגריתמיים והנקודות מצוירות בשקיפות גבוהה.
* **אמצע** - ארבע קטגוריות ה-cut vertices כספירות. עמודת `bus_only` היא החשובה: אלו פגיעויות שניתוח כל האופנים ב-notebook 03 אינו יכול לראות.
* **ימין** - שתי התפלגויות הדרגות על צירי log-log. אם רשת האוטובוסים אכן מהווה כמעט העתק של הרשת הארצית, שתי העקומות אמורות להיות כמעט בלתי ניתנות להבחנה; כל הפרדה בקצה הדרגות הגבוהות היא מסופי המעבר הרב-אופניים.

In [ ]:
# --- Comparison figure ----------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

x = np.array([deg_bus[n] for n in shared_sorted], dtype=float)
y = np.array([deg_all[n] for n in shared_sorted], dtype=float)
axes[0].scatter(x, y, s=6, alpha=0.15, color='#2563eb', linewidths=0)
lim = max(x.max(), y.max())
axes[0].plot([1, lim], [1, lim], color='#111827', linestyle='--', linewidth=1,
             label='y = x (bus is the whole story)')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_xlabel('Degree in the bus-only network')
axes[0].set_ylabel('Degree in the all-mode network')
axes[0].set_title(f'Degree agreement over {len(shared_sorted):,} shared stops\n'
                  f'Spearman rho = {rho_degree:.3f}')
axes[0].legend(fontsize=9)

buckets = ['both', 'bus_only', 'all_modes_only', 'absent_from_bus_network']
counts = [len(ap_both), len(ap_bus_only), len(ap_all_only), len(ap_absent)]
bars = axes[1].bar(range(len(buckets)), counts,
                   color=['#94a3b8', '#dc2626', '#f59e0b', '#64748b'])
axes[1].set_xticks(range(len(buckets)))
axes[1].set_xticklabels(buckets, rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel('Number of stations')
axes[1].set_title('Where the cut vertices agree and disagree')
for bar, val in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:,}',
                 ha='center', va='bottom', fontsize=10)
axes[1].margins(y=0.15)

for label, degmap, colour in [('bus only', deg_bus, '#2563eb'),
                              ('all modes', deg_all, '#dc2626')]:
    counts_by_deg = pd.Series(list(degmap.values())).value_counts().sort_index()
    counts_by_deg = counts_by_deg[counts_by_deg.index > 0]
    axes[2].scatter(counts_by_deg.index, counts_by_deg.to_numpy(), s=14, alpha=0.7,
                    color=colour, label=label)
axes[2].set_xscale('log')
axes[2].set_yscale('log')
axes[2].set_xlabel('Degree')
axes[2].set_ylabel('Number of stations')
axes[2].set_title('Degree distribution, bus vs all modes')
axes[2].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'bus_vs_allmode.png', dpi=FIG_DPI)
plt.show()

## 17. אוטובוס מבוסס-ביקוש (`route_type = 715`), מדווח בנפרד

הפיד מכיל אופן שני דמוי אוטובוס: 14 קווים / 458 נסיעות של שירות מבוסס-ביקוש. הוא *אינו* ממוזג לתוך רשת האוטובוסים שלעיל, משתי סיבות. מבנית, ה"קטעים" שלו מתארים דפוס שירות גמיש ולא קו קבוע, ולכן קשת שם אינה משמעה אותו הדבר כמו קשת בקו מתוזמן. מעשית, ב-458 נסיעות מתוך 412k הוא לא היה יכול לשנות שום אגרגט, אך היה יכול להוסיף בחשאי תחנות וקשתות שהדוח היה נדרש להסביר לאחר מכן.

לפיכך אנו מודדים אותו בפני עצמו - בכמה תחנות הוא נוגע, כמה קטעים יש בו, עד כמה הוא מקוטע, וכמה מתחנותיו כבר מקבלות שירות אוטובוס רגיל - וכותבים את התוצאה לטבלה בת שורה אחת משלו. אם החפיפה עם רשת האוטובוסים הרגילה גבוהה, שכבה זו אינה מוסיפה למעשה דבר לקשירות; אם היא נמוכה, היא משרתת מקומות שהאוטובוס המתוזמן אינו מגיע אליהם, וזהו ממצא בפני עצמו עבור ה-notebooks העוסקים בשוויוניות.

In [ ]:
# --- The demand-responsive layer, kept separate ---------------------------
drt_counts = edge_counts.get('drt', {})
if drt_counts:
    G_drt, D_drt = build_graphs(drt_counts)
    drt_components = sorted(nx.connected_components(G_drt), key=len, reverse=True)
    drt_nodes = set(G_drt.nodes())
    drt_summary = {
        'route_type': DRT_ROUTE_TYPE,
        'mode_label': MODE_LABELS[DRT_ROUTE_TYPE],
        'trips_streamed': stream_stats.get('drt_trips_streamed', 0),
        'stops': G_drt.number_of_nodes(),
        'directed_edges': D_drt.number_of_edges(),
        'undirected_edges': G_drt.number_of_edges(),
        'connected_components': len(drt_components),
        'largest_component_nodes': len(drt_components[0]),
        'stops_also_served_by_bus': len(drt_nodes & bus_nodes),
        'stops_unique_to_this_mode': len(drt_nodes - bus_nodes),
        'share_of_stops_shared_with_bus': round(len(drt_nodes & bus_nodes)
                                                / len(drt_nodes), 4),
        'edges_shared_with_bus': len({tuple(sorted(e)) for e in G_drt.edges()}
                                     & bus_edge_set),
    }
else:
    G_drt = nx.Graph()
    drt_summary = {'route_type': DRT_ROUTE_TYPE,
                   'mode_label': MODE_LABELS[DRT_ROUTE_TYPE],
                   'trips_streamed': 0, 'stops': 0, 'directed_edges': 0,
                   'undirected_edges': 0, 'connected_components': 0,
                   'largest_component_nodes': 0, 'stops_also_served_by_bus': 0,
                   'stops_unique_to_this_mode': 0,
                   'share_of_stops_shared_with_bus': None,
                   'edges_shared_with_bus': 0}
    print('No demand-responsive segments were found in the feed.')

pd.DataFrame([drt_summary]).to_csv(TABLES / 'demand_responsive_summary.csv',
                                   index=False, encoding='utf-8-sig')
for key, value in drt_summary.items():
    print(f'{key:34s} {value}')
print('\nsaved:', TABLES / 'demand_responsive_summary.csv')

## 18. שמירת תוצרי השלב

שלושה דברים נכתבים כאן, כך ששום שלב במורד הזרם לא יידרש לקרוא שוב בזרימה 816 MB של פיד:

* `bus_graph_undirected.pkl` - גרף האוטובוסים עצמו, עם מאפייני הצמתים מצורפים. Pickles שבירים מבחינת גרסאות, ולכן אותו מידע מיוצא גם כ-CSV פשוט.
* `tables/bus_edges.csv` - הקטעים המכוונים של האוטובוסים (`from_stop, to_stop, trip_frequency`), הצורה הניידת של הגרף.
* `bus_summary.json` - כל מספר מפתח שהופק לעיל בתוספת הקבועים שהפיקו אותו, כך שתמיד ניתן להתחקות אחר תוצאה עד להגדרות שיצרו אותה.

קבצים נכתבים **אך ורק** תחת `outputs/nb/15_bus_network/`.

In [ ]:
# --- Persist the graph, the edge list and the summary ---------------------
with open(STAGE / 'bus_graph_undirected.pkl', 'wb') as f:
    pickle.dump(G_bus, f)

bus_edges_df = pd.DataFrame([{'from_stop': u, 'to_stop': v,
                              'trip_frequency': int(data['weight'])}
                             for u, v, data in D_bus.edges(data=True)])
bus_edges_df.to_csv(TABLES / 'bus_edges.csv', index=False, encoding='utf-8-sig')

resilience_headline = {
    row['strategy'] + f"_share_at_{row['removed']}_removed":
        row['largest_component_share']
    for _, row in resilience.iterrows()
    if row['removed'] in (100, 1000, 3000)
}

bus_summary = {
    'constants': {
        'BUS_ROUTE_TYPE': BUS_ROUTE_TYPE,
        'DRT_ROUTE_TYPE': DRT_ROUTE_TYPE,
        'K_BETWEENNESS': K_BETWEENNESS,
        'COMPUTE_ALLMODE_BETWEENNESS': COMPUTE_ALLMODE_BETWEENNESS,
        'SEED': SEED,
        'RANDOM_TRIALS': RANDOM_TRIALS,
        'REMOVAL_COUNTS': REMOVAL_COUNTS,
        'TOP_N': TOP_N,
    },
    'mode_inventory': inventory.to_dict('records'),
    'stream_stats': stream_stats,
    'bus_structure': bus_stats,
    'all_mode_structure': all_stats,
    'bus_vs_all_mode': overlap,
    'resilience_headline': resilience_headline,
    'demand_responsive': drt_summary,
}
with open(STAGE / 'bus_summary.json', 'w', encoding='utf-8') as f:
    json.dump(bus_summary, f, ensure_ascii=False, indent=2, default=str)

written = [STAGE / 'bus_graph_undirected.pkl', STAGE / 'bus_summary.json',
           TABLES / 'bus_station_metrics.csv', TABLES / 'bus_edges.csv',
           TABLES / 'top_bus_stations.csv', TABLES / 'bus_articulation_points.csv',
           TABLES / 'bus_bridges.csv', TABLES / 'bus_resilience.csv',
           TABLES / 'bus_vs_allmode_summary.csv',
           TABLES / 'articulation_point_comparison.csv',
           TABLES / 'demand_responsive_summary.csv',
           FIGURES / 'top_bus_stations.png', FIGURES / 'bus_resilience_curves.png',
           FIGURES / 'bus_vs_allmode.png']
print('Written:')
for p in written:
    flag = 'ok ' if p.exists() else 'MISSING '
    size = f'{p.stat().st_size / 1024:,.0f} KB' if p.exists() else ''
    print(f'  {flag}{p}  {size}')

## מסקנות

יש לקרוא אותן מול המספרים שהודפסו לעיל; ה-notebook כתוב כך שכל טענה כאן ניתנת לבדיקה מתוך טבלה שהוא זה עתה כתב.

* **האוטובוס אכן מהווה כמעט את כל הרשת - וה-notebook בודק זאת במקום להניח זאת.** מלאי האופנים בסעיף 6 מאשר ש-`route_type = 3` נושא כ-97% מכלל הנסיעות המתוזמנות, וסעיף 15 מכמת מה משמעות הדבר מבחינה מבנית: נתוני כיסוי הצמתים וכיסוי הקשתות הלא-מכוונות, מתאם Spearman בין דרגת האוטובוס לדרגת כל האופנים, והשיעור של התחנות שדרגתן *זהה* בשני הגרפים. היכן ששיעור הדרגות הזהות גבוה, התחנות המתאימות הן כאלו שהשכבה הלא-אוטובוסית אינה נוגעת בהן לעולם, וכל תוצאה של כל האופנים לגביהן מועברת לאוטובוס ללא שינוי.
* **אך המילה "כמעט" נושאת משקל, והיא מתגלה בדיוק ב-cut vertices.** קטגוריית `bus_only` ב-`articulation_point_comparison.csv` מונה תחנות ש**אינן** נקודות כשל בודדות בגרף הארצי אך **כן** כאלו באוטובוס בלבד - מקומות שבהם קישור רכבת או רכבת קלה סיפק בחשאי את המסלול החלופי היחיד. אלו בלתי נראות ל-notebook 03 והן התשובה הקונקרטית לשאלה "האם הסרת הרכבות משנה אילו תחנות הן cut vertices?". באופן סימטרי, הקטגוריות `all_modes_only` ו-`absent_from_bus_network` הן cut vertices הקיימים רק משום שתחנות לא-אוטובוסיות נמצאות בגרף. אם שתי קטגוריות אי-ההסכמה יוצאות קטנות ביחס ל-`both`, המסקנה הכנה היא ששתי הרשתות מסכימות על *רוב* נקודות הכשל הבודדות ובכל זאת חלוקות על רשימה ספציפית וניתנת לשיום - והרשימה הזו, ולא האגרגט, היא הפלט השימושי.
* **הסרה ממוקדת גוברת על הסרה אקראית בפער ניכר, כמצופה מרשת מרחבית בעלת זנב כבד** - אך יש לשים לב לשתי ההסתייגויות שנוסחו בסעיף 13: דירוגי התקיפה סטטיים (לעולם אינם מחושבים מחדש לאחר הסרה), ולכן העקומות הממוקדות *מקטינות בתיאור* את מה שתוקף אדפטיבי היה יכול לעשות, וסדר ה-betweenness נשען על מדגם בן 300 pivots. הבסיס האקראי הוא האמין יותר מבין שתי העקומות, משום שאינו זקוק לדירוג כלשהו.
* **Betweenness מדגמי הוא אומדן, והזנב שלו הוא רעש.** עם `K_BETWEENNESS = 300` pivots מתוך כ-29k מקורות אפשריים, התחנות המובילות אמינות והערכים הקטנים אינם. כל notebook במורד הזרם הצורך `approx_betweenness` מתוך `bus_station_metrics.csv` צריך להשתמש בו לדירוג ראש ההתפלגות, ולא להשוואות עדינות בין תחנות אמצע-טבלה. הבדיקה הצולבת האופציונלית מול notebook 04 בסעיף 14 היא בדיוק זו: שני מדגמים בלתי תלויים של אותו גודל, ולכן Spearman rho קטן מ-1 הוא צפוי ומהווה מדד לרעש הדגימה, לא לאי-הסכמה.
* **קריטיות האוטובוס היא טופולוגית, לא מבוססת ביקוש.** כל מה שנעשה כאן סופר נסיעות וקטעים; איש אינו עולה לאוטובוס במודל הזה. cut vertex בדרגה 2 בקצה ענף כפרי ומסוף מעבר בדרגה 30 נספרים שניהם כ"נקודת חיתוך אחת" עד שמצורף משקל ביקוש, וזו התוספת של ה-notebooks העוסקים בסוציו-אקונומיה ובשקלול לפי ביקוש. עמודת הדרגה ב-`bus_articulation_points.csv` היא המסנן הראשוני והגס.
* **שירות מבוסס-ביקוש (`route_type = 715`) הוא שגיאת עיגול במונחי קשירות** - 14 קווים ו-458 נסיעות - והוא מדווח בטבלה נפרדת משלו במקום להיות מקופל לתוך גרף האוטובוסים. המספר המעניין היחיד שלו הוא שיעור התחנות שלו שהאוטובוס הרגיל כבר משרת: שיעור נמוך היה משמעו שהוא מגיע למקומות שהאוטובוס המתוזמן אינו מגיע אליהם, וזה חשוב לשוויוניות גם אם אינו יכול להיות משמעותי טופולוגית בנפח כזה.
* **מה ה-notebook הזה אינו יכול לומר לכם.** מדובר בתצלום מצב סטטי יחיד של כל תקופת הפיד, ללא פילוח לפי שעות היום (notebooks 19-20), ללא זמני נסיעה (notebook 18), וללא כל מושג של נוסע שיכול ללכת 100 מ' לתחנה אחרת - שתי תחנות משני צדי אותו צומת הן צמתים נפרדים ובלתי מחוברים אלא אם אוטובוס אכן נוסע ביניהן. הנחה אחרונה זו גורמת לרשת להיראות שברירית יותר משהיא בשטח, והיא חלה על ספירות ה-cut vertices כאן בדיוק כפי שהיא חלה ב-notebook 03.